# Ablation Study — Multichannel Sarcasm Detection — Google Colab

**Cara pakai:**
1. Pastikan folder `multi_channel_method/` sudah di Google Drive (`My Drive/multi_channel_method/`)
2. Pilih **Runtime → Change runtime type → T4 GPU**
3. Jalankan cell dari atas ke bawah
4. Jika sesi Colab putus, cukup mount ulang Drive (Cell 1-2), lalu lanjut dari cell yang terakhir dijalankan

**7 Variant ablation yang dijalankan:**
| Variant | Deskripsi |
|---|---|
| `full` | Baseline: BERT + ADGCN + dep & sentic graph |
| `bert_only` | Hanya BERT CLS, tanpa ADGCN |
| `adgcn_only` | Hanya ADGCN (BiLSTM + dep & sentic alternating), tanpa BERT |
| `dep_only` | BERT + ADGCN, semua GCN layer pakai dependency graph |
| `sentic_only` | BERT + ADGCN, semua GCN layer pakai sentic graph |
| `gcn_dep_only` | ADGCN saja dengan dep graph (semua 6 layer), tanpa BERT |
| `gcn_sentic_only` | ADGCN saja dengan sentic graph (semua 6 layer), tanpa BERT |

**Catatan default baru (per perbaikan bug):**
- `--early_stopping_patience` default sekarang `8` (sebelumnya 3)
- Focal Loss `alpha=0.75` (sebelumnya 0.25 yang terbalik)
- Seed di-reset di awal setiap variant untuk reproducibility

## Cell 1 — Environment Setup (Colab / Kaggle)

> **Kaggle**: ubah  sesuai nama dataset di   
> **Colab**: tidak perlu mengubah apapun, Drive akan di-mount otomatis


In [ ]:
import os, sys, shutil

# Auto-detect environment: Kaggle vs Colab
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # ── KAGGLE SETUP ──────────────────────────────────────────────────────
    # Cari nama folder dataset di /kaggle/input/ lalu isi KAGGLE_INPUT_PATH
    print("Input datasets tersedia:", os.listdir('/kaggle/input'))

    # Ubah KAGGLE_INPUT_PATH sesuai nama dataset kamu
    KAGGLE_INPUT_PATH = '/kaggle/input/NAMA-DATASET/multi_channel_method'
    WORK_DIR = '/kaggle/working/multi_channel_method'

    if not os.path.exists(WORK_DIR):
        print(f'Menyalin project ke {WORK_DIR} ...')
        shutil.copytree(KAGGLE_INPUT_PATH, WORK_DIR, dirs_exist_ok=False)
        print('Selesai.')
    else:
        print(f'{WORK_DIR} sudah ada, skip copy.')

    os.chdir(WORK_DIR)
    print(f'Kaggle mode: working dir = {WORK_DIR}')

else:
    # ── COLAB SETUP ───────────────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/multi_channel_method'
    os.chdir(PROJECT_PATH)
    print(f'Colab mode: working dir = {PROJECT_PATH}')

print('Files:', os.listdir('.'))


## Cell 2 — Konfirmasi Working Directory


In [ ]:
import os
print('Working dir:', os.getcwd())
print('Files found:', os.listdir('.'))


## Cell 3 — Install Dependencies & Cek GPU

In [ ]:
!pip install -q --upgrade transformers sentencepiece
!nvidia-smi

## Cell 4 — Buat Direktori Output

In [ ]:
import os

VARIANTS = ['full', 'bert_only', 'adgcn_only', 'dep_only', 'sentic_only',
            'gcn_dep_only', 'gcn_sentic_only']

for dataset in ['reddit', 'twitter']:
    for variant in VARIANTS:
        os.makedirs(f'checkpoints/ablation/{dataset}/{variant}', exist_ok=True)

os.makedirs('logs', exist_ok=True)
print('Direktori berhasil dibuat.')

## Cell 5 — Verifikasi File Data

In [ ]:
import os

all_ok = True
for dataset in ['reddit', 'twitter']:
    for split in ['train', 'test', 'validation']:
        for ext in ['.csv', '.csv.graph.new', '.csv.sentic']:
            path = f'real_data/{dataset}/{split}{ext}'
            status = 'OK' if os.path.exists(path) else 'MISSING!'
            if status == 'MISSING!':
                all_ok = False
            print(f'[{dataset}] {split}{ext}: {status}')

print()
if all_ok:
    print('Semua file tersedia. Siap ablation study!')
else:
    print('Ada file yang hilang! Periksa upload Google Drive.')

---
## DATASET: REDDIT

Jalankan cell 6–10 secara berurutan. Tiap cell = satu variant.  
Jika sesi putus di tengah, cukup lanjut dari cell variant yang belum selesai.  
Hasil otomatis ter-append ke `ablation_results_reddit.csv`.

## Cell 6 — Reddit: Variant `full` (baseline)

In [ ]:
!python train_ablation.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/ablation/reddit \
    --results_csv      ablation_results_reddit.csv \
    --ablation         full \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_reddit_full.log

## Cell 7 — Reddit: Variant `bert_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/ablation/reddit \
    --results_csv      ablation_results_reddit.csv \
    --ablation         bert_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_reddit_bert_only.log

## Cell 8 — Reddit: Variant `adgcn_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/ablation/reddit \
    --results_csv      ablation_results_reddit.csv \
    --ablation         adgcn_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_reddit_adgcn_only.log

## Cell 9 — Reddit: Variant `dep_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/ablation/reddit \
    --results_csv      ablation_results_reddit.csv \
    --ablation         dep_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_reddit_dep_only.log

## Cell 10 — Reddit: Variant `sentic_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/ablation/reddit \
    --results_csv      ablation_results_reddit.csv \
    --ablation         sentic_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_reddit_sentic_only.log

## Cell 10a — Reddit: Variant `gcn_dep_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/ablation/reddit \
    --results_csv      ablation_results_reddit.csv \
    --ablation         gcn_dep_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_reddit_gcn_dep_only.log

## Cell 10b — Reddit: Variant `gcn_sentic_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/ablation/reddit \
    --results_csv      ablation_results_reddit.csv \
    --ablation         gcn_sentic_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_reddit_gcn_sentic_only.log

## Cell 11 — Reddit: Lihat Hasil Semua Variant

In [ ]:
import pandas as pd

df = pd.read_csv('ablation_results_reddit.csv')
print('=== ABLATION RESULTS — REDDIT ===')
print(df.to_string(index=False))

---
## DATASET: TWITTER

Sama seperti Reddit — jalankan tiap cell satu per satu.

## Cell 12 — Twitter: Variant `full` (baseline)

In [ ]:
!python train_ablation.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/ablation/twitter \
    --results_csv      ablation_results_twitter.csv \
    --ablation         full \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_twitter_full.log

## Cell 13 — Twitter: Variant `bert_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/ablation/twitter \
    --results_csv      ablation_results_twitter.csv \
    --ablation         bert_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_twitter_bert_only.log

## Cell 14 — Twitter: Variant `adgcn_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/ablation/twitter \
    --results_csv      ablation_results_twitter.csv \
    --ablation         adgcn_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_twitter_adgcn_only.log

## Cell 15 — Twitter: Variant `dep_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/ablation/twitter \
    --results_csv      ablation_results_twitter.csv \
    --ablation         dep_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_twitter_dep_only.log

## Cell 16 — Twitter: Variant `sentic_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/ablation/twitter \
    --results_csv      ablation_results_twitter.csv \
    --ablation         sentic_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_twitter_sentic_only.log

## Cell 16a — Twitter: Variant `gcn_dep_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/ablation/twitter \
    --results_csv      ablation_results_twitter.csv \
    --ablation         gcn_dep_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_twitter_gcn_dep_only.log

## Cell 16b — Twitter: Variant `gcn_sentic_only`

In [ ]:
!python train_ablation.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/ablation/twitter \
    --results_csv      ablation_results_twitter.csv \
    --ablation         gcn_sentic_only \
    --epochs           100 \
    --batch_size       32 \
    --seed             42 \
    2>&1 | tee logs/ablation_twitter_gcn_sentic_only.log

## Cell 17 — Twitter: Lihat Hasil Semua Variant

In [ ]:
import pandas as pd

df = pd.read_csv('ablation_results_twitter.csv')
print('=== ABLATION RESULTS — TWITTER ===')
print(df.to_string(index=False))

## Cell 18 — Lihat Log Jika Ada Error

In [ ]:
# Ganti nama file sesuai variant yang ingin dilihat
!tail -80 logs/ablation_reddit_full.log